In [2]:
from huggingface_hub import snapshot_download
import glob

path = snapshot_download("Mouwiya/cfpb-consumer-complaints", repo_type="dataset", allow_patterns="*.parquet")
files = sorted(glob.glob(path + "/**/*.parquet", recursive=True))
print(len(files), "files downloaded")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 67 files:   0%|          | 0/67 [00:00<?, ?it/s]

67 files downloaded


In [3]:
import pandas as pd
import pyarrow.parquet as pq

cols = ["product", "consumer_complaint_narrative", "date_received"]
cutoff = pd.Timestamp("2024-01-01", tz="UTC")
parts = []

for f in files:
    d = pq.read_table(f, columns=cols).to_pandas()
    d = d[d["consumer_complaint_narrative"].notna()]
    d = d[pd.to_datetime(d["date_received"], utc=True, errors="coerce") >= cutoff]
    parts.append(d.sample(frac=0.25, random_state=42))

df = pd.concat(parts, ignore_index=True)
df = df.sample(n=min(60000, len(df)), random_state=42).reset_index(drop=True)
df = df.rename(columns={"consumer_complaint_narrative": "text"})[["text", "product"]]
print(df.shape)

(60000, 2)


In [4]:
print(df["product"].value_counts())
print(df["text"].str.split().str.len().describe())
print(df["text"].duplicated().sum(), "duplicate complaints")
print(df["text"].iloc[0][:400])
df.to_parquet("/content/complaints.parquet", index=False)

product
Credit reporting or other personal consumer reports        43555
Debt collection                                             5659
Checking or savings account                                 2774
Credit card                                                 2646
Money transfer, virtual currency, or money service          2322
Mortgage                                                     875
Vehicle loan or lease                                        700
Student loan                                                 634
Payday loan, title loan, personal loan, or advance loan      447
Prepaid card                                                 250
Debt or credit management                                    138
Name: count, dtype: int64
count    60000.000000
mean       170.415533
std        211.087544
min          1.000000
25%         58.000000
50%        112.000000
75%        207.000000
max       5674.000000
Name: text, dtype: float64
17223 duplicate complaints
Finding unauthorized 

Clean-up

In [5]:
import pandas as pd

df = pd.read_parquet("/content/complaints.parquet")
n0 = len(df)

# 1. remove redaction tokens (XXXX, XX/XX/XXXX) and extra spaces
df["text"] = (df["text"].str.replace(r"\b(?:XX/)*X{2,}\b", " ", regex=True)
                        .str.split().str.join(" "))

# 2. drop complaints too short to classify
df = df[df["text"].str.split().str.len() >= 10]

# 3. drop texts that appear with more than one product label
n_labels = df.groupby("text")["product"].nunique()
df = df[~df["text"].isin(n_labels[n_labels > 1].index)]

# 4. drop exact duplicates, keeping one copy
df = df.drop_duplicates("text")

# 5. drop products with fewer than 100 complaints
counts = df["product"].value_counts()
df = df[df["product"].isin(counts[counts >= 100].index)].reset_index(drop=True)

print(n0, "->", len(df), "rows")
print(df["product"].value_counts())
df.to_parquet("/content/complaints_clean.parquet", index=False)

60000 -> 40428 rows
product
Credit reporting or other personal consumer reports        25984
Debt collection                                             4482
Checking or savings account                                 2659
Credit card                                                 2547
Money transfer, virtual currency, or money service          1727
Mortgage                                                     873
Vehicle loan or lease                                        696
Student loan                                                 631
Payday loan, title loan, personal loan, or advance loan      443
Prepaid card                                                 248
Debt or credit management                                    138
Name: count, dtype: int64


EDA


In [6]:
# Class size comparison
f = pd.read_parquet("/content/complaints_clean.parquet")

counts = df["product"].value_counts()
print((counts / len(df) * 100).round(1))
print("largest class is", round(counts.max() / counts.min()), "x the smallest")

product
Credit reporting or other personal consumer reports        64.3
Debt collection                                            11.1
Checking or savings account                                 6.6
Credit card                                                 6.3
Money transfer, virtual currency, or money service          4.3
Mortgage                                                    2.2
Vehicle loan or lease                                       1.7
Student loan                                                1.6
Payday loan, title loan, personal loan, or advance loan     1.1
Prepaid card                                                0.6
Debt or credit management                                   0.3
Name: count, dtype: float64
largest class is 188 x the smallest


In [7]:
# Complaint lengths
df["words"] = df["text"].str.split().str.len()
print(df["words"].describe().round(0))
print(df.groupby("product")["words"].median().sort_values())

count    40428.0
mean       189.0
std        229.0
min         10.0
25%         64.0
50%        126.0
75%        239.0
max       5407.0
Name: words, dtype: float64
product
Money transfer, virtual currency, or money service         106.0
Debt or credit management                                  114.0
Debt collection                                            118.0
Credit reporting or other personal consumer reports        119.0
Prepaid card                                               128.0
Checking or savings account                                143.0
Vehicle loan or lease                                      161.5
Payday loan, title loan, personal loan, or advance loan    162.0
Student loan                                               165.0
Credit card                                                166.0
Mortgage                                                   217.0
Name: words, dtype: float64


In [8]:
# Tokens needed
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample = df["text"].sample(3000, random_state=42).tolist()
lens = pd.Series([len(ids) for ids in tok(sample)["input_ids"]])
for n in (128, 256, 512):
    print(f"{n} tokens covers {(lens <= n).mean():.0%} of complaints")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (621 > 512). Running this sequence through the model will result in indexing errors


128 tokens covers 41% of complaints
256 tokens covers 69% of complaints
512 tokens covers 91% of complaints


In [9]:
key = df["text"].str[:100].str.lower()
shared = key.duplicated(keep=False)
print(f"{shared.mean():.1%} of complaints share their first 100 characters with another")
print(shared.groupby(df["product"]).mean().mul(100).round(1))

22.6% of complaints share their first 100 characters with another
product
Checking or savings account                                 3.2
Credit card                                                 2.2
Credit reporting or other personal consumer reports        32.3
Debt collection                                             4.6
Debt or credit management                                   2.2
Money transfer, virtual currency, or money service         22.1
Mortgage                                                    0.1
Payday loan, title loan, personal loan, or advance loan     0.5
Prepaid card                                                0.0
Student loan                                                0.6
Vehicle loan or lease                                       0.7
Name: text, dtype: float64


In [10]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

df = pd.read_parquet("/content/complaints_clean.parquet")

def group_split(data):
    groups = data["text"].str[:100].str.lower()
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    big, small = next(cv.split(data, data["product"], groups))
    return data.iloc[big], data.iloc[small]

trainval, test = group_split(df)
train, val = group_split(trainval)

In [11]:
key = lambda d: set(d["text"].str[:100].str.lower())
print("shared groups  train/val:", len(key(train) & key(val)),
      "| train/test:", len(key(train) & key(test)),
      "| val/test:", len(key(val) & key(test)))
print(pd.DataFrame({"train": train["product"].value_counts(),
                    "val": val["product"].value_counts(),
                    "test": test["product"].value_counts()}))

for name, d in [("train", train), ("val", val), ("test", test)]:
    d.to_parquet(f"/content/{name}.parquet", index=False)

shared groups  train/val: 0 | train/test: 0 | val/test: 0
                                                    train   val  test
product                                                              
Credit reporting or other personal consumer rep...  16905  4188  4891
Debt collection                                      2850   697   935
Checking or savings account                          1708   425   526
Credit card                                          1615   416   516
Money transfer, virtual currency, or money service   1149   309   269
Mortgage                                              579   126   168
Vehicle loan or lease                                 462   109   125
Student loan                                          434    79   118
Payday loan, title loan, personal loan, or adva...    272    69   102
Prepaid card                                          156    41    51
Debt or credit management                              90    19    29


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report

base = make_pipeline(
    TfidfVectorizer(min_df=3, sublinear_tf=True),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
base.fit(train["text"], train["product"])
pred = base.predict(val["text"])

majority = train["product"].mode()[0]
print("always-majority macro-F1:", round(f1_score(val["product"], [majority] * len(val), average="macro", zero_division=0), 3))
print("baseline macro-F1       :", round(f1_score(val["product"], pred, average="macro"), 3))
print(classification_report(val["product"], pred, zero_division=0))

always-majority macro-F1: 0.071
baseline macro-F1       : 0.642
                                                         precision    recall  f1-score   support

                            Checking or savings account       0.74      0.79      0.76       425
                                            Credit card       0.60      0.73      0.66       416
    Credit reporting or other personal consumer reports       0.96      0.84      0.90      4188
                                        Debt collection       0.59      0.78      0.67       697
                              Debt or credit management       0.11      0.21      0.14        19
     Money transfer, virtual currency, or money service       0.81      0.81      0.81       309
                                               Mortgage       0.73      0.91      0.81       126
Payday loan, title loan, personal loan, or advance loan       0.39      0.54      0.45        69
                                           Prepaid card       

In [13]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

results = []
for ngram in [(1, 1), (1, 2)]:
    tfidf = TfidfVectorizer(ngram_range=ngram, min_df=3, sublinear_tf=True)
    Xtr = tfidf.fit_transform(train["text"])
    Xva = tfidf.transform(val["text"])
    for C in [1, 10]:
        for cw in ["balanced", None]:
            m = LogisticRegression(C=C, class_weight=cw, max_iter=1000).fit(Xtr, train["product"])
            f1 = f1_score(val["product"], m.predict(Xva), average="macro")
            results.append((ngram, C, cw, round(f1, 3)))
            print(results[-1])

((1, 1), 1, 'balanced', 0.642)
((1, 1), 1, None, 0.618)
((1, 1), 10, 'balanced', 0.644)
((1, 1), 10, None, 0.652)
((1, 2), 1, 'balanced', 0.669)
((1, 2), 1, None, 0.573)
((1, 2), 10, 'balanced', 0.659)
((1, 2), 10, None, 0.645)


In [14]:
from sklearn.metrics import confusion_matrix

ngram, C, cw, _ = max(results, key=lambda r: r[3])
tfidf = TfidfVectorizer(ngram_range=ngram, min_df=3, sublinear_tf=True)
Xtr = tfidf.fit_transform(train["text"])
Xva = tfidf.transform(val["text"])
model = LogisticRegression(C=C, class_weight=cw, max_iter=1000).fit(Xtr, train["product"])
pred = model.predict(Xva)

labels = sorted(train["product"].unique())
print(pd.Series(f1_score(val["product"], pred, average=None, labels=labels), index=labels).round(2).sort_values())

cm = pd.DataFrame(confusion_matrix(val["product"], pred, labels=labels), index=labels, columns=labels)
pairs = cm.stack().rename("n").reset_index()
pairs.columns = ["true", "predicted", "n"]
print(pairs[pairs["true"] != pairs["predicted"]].nlargest(8, "n").to_string(index=False))

Debt or credit management                                  0.25
Payday loan, title loan, personal loan, or advance loan    0.48
Vehicle loan or lease                                      0.62
Prepaid card                                               0.63
Credit card                                                0.67
Debt collection                                            0.67
Student loan                                               0.73
Checking or savings account                                0.79
Mortgage                                                   0.80
Money transfer, virtual currency, or money service         0.82
Credit reporting or other personal consumer reports        0.90
dtype: float64
                                               true                                           predicted   n
Credit reporting or other personal consumer reports                                     Debt collection 335
Credit reporting or other personal consumer reports              

In [15]:
mask = pred != val["product"].values
wrong = val[mask].assign(predicted=pred[mask])
hard = wrong[wrong["product"] == "Debt or credit management"].head(4)
for _, r in hard.iterrows():
    print(f"TRUE: {r['product']} | PREDICTED: {r['predicted']}\n{r['text'][:300]}\n")

TRUE: Debt or credit management | PREDICTED: Checking or savings account
FDR is a voluntary program. They refuse to allow me to cancel. They charged and overdrew my checking account! I've called them 3 times. I never received an email confirmation that I've requested to close my account. They are making a bad day even worse. I don't know what else to do at this point.

TRUE: Debt or credit management | PREDICTED: Checking or savings account
I was charged upfront fees and other random fees and no services given

TRUE: Debt or credit management | PREDICTED: Credit reporting or other personal consumer reports
Under the Fair Credit Reporting Act ( 15 U.S.C. 1681 ), consumer reporting agencies are required to ensure maximum possible accuracy of information reported. Reporting a closed account as open violates this standard. I am requesting that you conduct a reinvestigation and delete or correct this accou

TRUE: Debt or credit management | PREDICTED: Debt collection
I was contacted by Sunr

In [16]:
import joblib
joblib.dump((tfidf, model), "/content/baseline.joblib")
base_val_pred = pred

In [17]:
import time, torch, numpy as np, pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

torch.manual_seed(42)
device = "cuda"
labels = sorted(train["product"].unique())
l2i = {l: i for i, l in enumerate(labels)}
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def encode(d):
    enc = tok(d["text"].tolist(), truncation=True, max_length=256)
    return [{"input_ids": i, "attention_mask": a, "labels": l2i[p]}
            for i, a, p in zip(enc["input_ids"], enc["attention_mask"], d["product"])]

def loader(items, bs, shuffle):
    return DataLoader(items, batch_size=bs, shuffle=shuffle,
                      collate_fn=lambda b: tok.pad(b, return_tensors="pt"))

train_items, val_items = encode(train), encode(val)
val_y = val["product"].map(l2i).values

In [18]:
bert = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=len(labels)).to(device)

y = train["product"].map(l2i).values
w = (len(y) / (len(labels) * np.bincount(y))) ** 0.5
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float, device=device))

EPOCHS = 3
train_dl = loader(train_items, 32, True)
steps = EPOCHS * len(train_dl)
opt = torch.optim.AdamW(bert.parameters(), lr=3e-5, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.amp.GradScaler("cuda")

@torch.no_grad()
def predict(items):
    bert.eval()
    out = []
    for b in loader(items, 128, False):
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(bert(input_ids=b["input_ids"].to(device),
                            attention_mask=b["attention_mask"].to(device)).logits.float().cpu())
    return torch.cat(out).numpy()

best, t0 = 0, time.time()
for epoch in range(EPOCHS):
    bert.train()
    for b in train_dl:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            logits = bert(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
        loss = loss_fn(logits.float(), b["labels"])
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(bert.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
    f1 = f1_score(val_y, predict(val_items).argmax(1), average="macro")
    print(f"epoch {epoch + 1}: val macro-F1 {f1:.3f} | {(time.time() - t0) / 60:.1f} min")
    if f1 > best:
        best = f1
        torch.save(bert.state_dict(), "/content/distilbert_best.pt")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1: val macro-F1 0.614 | 2.6 min
epoch 2: val macro-F1 0.642 | 5.2 min
epoch 3: val macro-F1 0.676 | 8.0 min


In [19]:
bert.load_state_dict(torch.load("/content/distilbert_best.pt"))
bert_pred = predict(val_items).argmax(1)
base_y = pd.Series(base_val_pred).map(l2i).values

print("macro-F1  baseline:", round(f1_score(val_y, base_y, average="macro"), 3),
      "| distilbert:", round(f1_score(val_y, bert_pred, average="macro"), 3))
print(pd.DataFrame({"n_val": np.bincount(val_y),
                    "baseline": f1_score(val_y, base_y, average=None),
                    "distilbert": f1_score(val_y, bert_pred, average=None)}, index=labels).round(2))

macro-F1  baseline: 0.669 | distilbert: 0.676
                                                    n_val  baseline  \
Checking or savings account                           425      0.79   
Credit card                                           416      0.67   
Credit reporting or other personal consumer rep...   4188      0.90   
Debt collection                                       697      0.67   
Debt or credit management                              19      0.25   
Money transfer, virtual currency, or money service    309      0.82   
Mortgage                                              126      0.80   
Payday loan, title loan, personal loan, or adva...     69      0.48   
Prepaid card                                           41      0.63   
Student loan                                           79      0.73   
Vehicle loan or lease                                 109      0.62   

                                                    distilbert  
Checking or savings account         

In [20]:
ckpt = "/content/distilbert_best.pt"   # use distilbert_best5.pt if the 5-epoch run won
# if Colab restarted: tfidf, model = joblib.load("/content/baseline.joblib")

test = pd.read_parquet("/content/test.parquet")
test_items = encode(test)
y = test["product"].map(l2i).values

bert.load_state_dict(torch.load(ckpt))
bert_p = predict(test_items).argmax(1)
base_p = pd.Series(model.predict(tfidf.transform(test["text"]))).map(l2i).values

L = list(range(len(labels)))
L10 = [i for i in L if labels[i] != "Debt or credit management"]
for name, p in [("baseline", base_p), ("distilbert", bert_p)]:
    print(f"{name:10} macro-F1 all 11: {f1_score(y, p, labels=L, average='macro'):.3f}"
          f" | without the tiny class: {f1_score(y, p, labels=L10, average='macro'):.3f}")

print(pd.DataFrame({"n_test": np.bincount(y),
                    "baseline": f1_score(y, base_p, labels=L, average=None),
                    "distilbert": f1_score(y, bert_p, labels=L, average=None)}, index=labels).round(2))

baseline   macro-F1 all 11: 0.659 | without the tiny class: 0.713
distilbert macro-F1 all 11: 0.666 | without the tiny class: 0.720
                                                    n_test  baseline  \
Checking or savings account                            526      0.78   
Credit card                                            516      0.69   
Credit reporting or other personal consumer rep...    4891      0.90   
Debt collection                                        935      0.69   
Debt or credit management                               29      0.12   
Money transfer, virtual currency, or money service     269      0.74   
Mortgage                                               168      0.83   
Payday loan, title loan, personal loan, or adva...     102      0.52   
Prepaid card                                            51      0.63   
Student loan                                           118      0.72   
Vehicle loan or lease                                  125      0.63   

   

In [21]:
rng = np.random.default_rng(42)
codes = pd.factorize(test["text"].str[:100].str.lower())[0]
G = codes.max() + 1

fb, fl = [], []
for _ in range(1000):
    w = rng.multinomial(G, np.ones(G) / G)[codes]
    fb.append(f1_score(y, base_p, labels=L, average="macro", sample_weight=w, zero_division=0))
    fl.append(f1_score(y, bert_p, labels=L, average="macro", sample_weight=w, zero_division=0))
fb, fl = np.array(fb), np.array(fl)

ci = lambda a: np.percentile(a, [2.5, 97.5]).round(3)
print("baseline   95% interval:", ci(fb))
print("distilbert 95% interval:", ci(fl))
print("difference (distilbert - baseline):", ci(fl - fb), "| DistilBERT wins in", f"{(fl > fb).mean():.0%} of resamples")

baseline   95% interval: [0.636 0.682]
distilbert 95% interval: [0.642 0.688]
difference (distilbert - baseline): [-0.015  0.028] | DistilBERT wins in 74% of resamples


In [22]:
import copy, os, time, joblib

t = time.time(); model.predict(tfidf.transform(test["text"])); base_ms = (time.time() - t) / len(test) * 1000

torch.cuda.synchronize(); t = time.time(); predict(test_items); torch.cuda.synchronize()
gpu_ms = (time.time() - t) / len(test) * 1000

cpu_bert = copy.deepcopy(bert).cpu().eval()
t = time.time()
with torch.no_grad():
    for b in loader(test_items[:200], 32, False):
        cpu_bert(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
cpu_ms = (time.time() - t) / 200 * 1000

joblib.dump((tfidf, model), "/tmp/b.joblib")
print(f"ms per complaint: baseline {base_ms:.2f} | DistilBERT on GPU {gpu_ms:.2f} | DistilBERT on CPU {cpu_ms:.1f}")
print(f"file size MB: baseline {os.path.getsize('/tmp/b.joblib') / 1e6:.0f} | DistilBERT {os.path.getsize(ckpt) / 1e6:.0f}")

ms per complaint: baseline 0.22 | DistilBERT on GPU 1.65 | DistilBERT on CPU 242.8
file size MB: baseline 15 | DistilBERT 268


In [23]:
import joblib, sklearn, numpy, platform

tfidf2, model2 = joblib.load("/content/baseline.joblib")
for t in ["I was charged a late fee on my credit card even though I paid on time",
          "My servicer will not apply my payments to my mortgage escrow account"]:
    print(model2.predict(tfidf2.transform([t]))[0])

open("/content/requirements.txt", "w").write(
    f"streamlit\nscikit-learn=={sklearn.__version__}\nnumpy=={numpy.__version__}\njoblib=={joblib.__version__}\n")
print("Python version for Streamlit settings:", platform.python_version())

Credit card
Mortgage
Python version for Streamlit settings: 3.13.16


In [24]:
%%writefile /content/app.py
import joblib, numpy as np, pandas as pd, streamlit as st

@st.cache_resource
def load():
    tfidf, model = joblib.load("baseline.joblib")
    return tfidf, model, np.array(tfidf.get_feature_names_out())

tfidf, model, words = load()

st.title("Complaint Triage")
st.caption("Predicts which financial product a complaint is about (TF-IDF + Logistic Regression).")
text = st.text_area("Paste a customer complaint", height=180)

if st.button("Classify") and text.strip():
    x = tfidf.transform([text])
    probs = model.predict_proba(x)[0]
    top = np.argsort(probs)[::-1][:3]
    st.table(pd.DataFrame({"product": model.classes_[top], "score": probs[top].round(2)}))

    contrib = x.toarray()[0] * model.coef_[top[0]]
    best = np.argsort(contrib)[::-1][:8]
    best = best[contrib[best] > 0]
    st.write("Words pushing toward the top prediction:", ", ".join(words[best]))
    st.caption("Scores rank products; they are not calibrated probabilities (class weights inflate small classes).")

Writing /content/app.py


In [25]:
!zip -j /content/nlp_app.zip /content/app.py /content/baseline.joblib /content/requirements.txt
from google.colab import files
files.download("/content/nlp_app.zip")

  adding: app.py (deflated 43%)
  adding: baseline.joblib (deflated 18%)
  adding: requirements.txt (deflated 4%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>